# HEILO Faísca 3.0 (200M): ~210 M parâmetros, do zero, em sessões diárias

**O que é:** a Faísca grande, **7× maior** que a 2.3 (30 M). Mesmo motor moderno (RoPE, RMSNorm, GeGLU, GQA, QK-norm), 100% da HEILO, treinada do zero. Nenhum peso de outro modelo.

**Duas novidades que atacam os erros da 2.x:**
- **Tokenizer novo, com algarismos separados** ("48" vira "4" + "8"). O antigo juntava números inteiros num pedaço só, e por isso as contas saíam erradas.
- **Ajuste de conversa corrigido**: guarda o ponto que *responde* melhor (antes parava no passo 100).

**Como funciona o dia a dia**
1. `Ambiente de execução → Alterar tipo → GPU` (se aparecer **L4 ou A100**, escolha: é bem mais rápido que a T4) e `Executar tudo`.
2. **1º dia:** ele treina o tokenizer novo e prepara a Wikipédia (~1 h) e depois treina.
3. Cada dia prepara **uma parte nova** de texto da web, treina até completar a sessão (`MINUTOS_SESSAO`, hoje ~12h20), **salva no Drive** e **desliga a máquina sozinha** para não gastar unidades. No dia seguinte continua de onde parou.
4. No fim de cada sessão mede a **régua fixa** (R1 perplexidade congelada + R2 gramática), que compara um dia com o outro de verdade.
5. Quando chegar a 100%, guarda uma **cópia congelada** do cérebro em `heilo_cerebros/` no Drive e faz sozinho o ajuste de conversa, a avaliação na mesma régua e o envio ao GitHub.
6. Quer testar antes do fim? Ponha `TESTAR_AGORA = True`: gera uma **prévia** (ex.: `v3.0-p40` = 40% do treino) e o pré-treino continua normalmente no dia seguinte.

**Quanto tempo:** a meta é ~3 bilhões de tokens. A célula 6 mostra a previsão real com a GPU do dia (T4 é a mais lenta; L4/A100 encurtam muito).

**Antes da 1ª vez:** rode `publicar_heilo.bat` no PC. O secret `GITHUB_TOKEN` é o mesmo de sempre.

In [ ]:
# 1) Configuração
import time; INICIO = time.time()
REPO = "hugolindo12/HELIO"
BRANCH = "main"
VERSAO = "v3.0"
MINUTOS_SESSAO = 740        # ~12h20 por sessão (com folga para salvar)
MAX_PARTES_WEB = 12         # 12 × ~250 M tokens de texto da web (+ Wikipédia)
TESTAR_AGORA = False        # True = gera uma prévia hoje (o pré-treino continua amanhã)
DESLIGAR_NO_FIM = True      # desliga a máquina do Colab no fim (para de gastar unidades)
import torch, os, sys, json, subprocess
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA — ative a GPU!"
mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print("GPU:", gpu, f"({mem:.0f} GB)")

In [ ]:
# 2) Drive (pasta própria: o tokenizer e o corpus são novos)
from google.colab import drive
drive.mount("/content/drive")
PASTA = "/content/drive/MyDrive/heilo_faisca200"
os.makedirs(PASTA, exist_ok=True)

In [ ]:
# 3) Código da HEILO (sempre o mais novo: pega também os exemplos que você aprovou)
from google.colab import userdata
TOKEN = userdata.get("GITHUB_TOKEN")
URL = f"https://{TOKEN}@github.com/{REPO}.git"
if not os.path.exists("/content/heilo"):
    subprocess.run(["git", "clone", "-b", BRANCH, URL, "/content/heilo"], check=True)
else:
    subprocess.run("cd /content/heilo && git stash -u -q; git pull -q --rebase", shell=True)
sys.path.insert(0, "/content")
!pip -q install datasets tokenizers
from heilo.training import aurora, faisca_grande, pretrain, seed_v1
from pathlib import Path
L = faisca_grande.lote_para_gpu(gpu, mem)
print("micro-lote", L["micro"], "× acumular", L["acumular"])

## Tokenizer e corpus (o corpus cresce uma parte por dia)

In [ ]:
# 4) Tokenizer novo (só no 1º dia) + Wikipédia + uma parte nova da web por sessão
t0 = time.time()
tok = faisca_grande.preparar_tokenizer(Path(PASTA))
print("vocabulário:", tok.vocab_size, "| exemplo:", [tok.tok.id_to_token(i) for i in tok.encode("6 vezes 8 é 48")])
wiki = Path(PASTA) / "wiki"
if not (wiki / "meta.json").exists():
    pretrain.preparar_corpus(seed_v1.textos_wikipedia(800_000_000), tok, wiki, max_tokens=10**12, log=lambda s: None)
web = Path(PASTA) / "web"
prontas = pretrain.pastas_corpus(web)
sobra = MINUTOS_SESSAO - (time.time() - INICIO) / 60
if len(prontas) < MAX_PARTES_WEB and not TESTAR_AGORA and (sobra > 150 or not prontas):
    try:
        aurora.preparar_proxima_parte(web, tok)
    except Exception as e:
        print("não consegui baixar texto da web hoje (continua com o que já tem):", e)
print(f"preparação: {(time.time()-t0)/60:.0f} min")
# copia para o disco rápido do Colab (ler do Drive direto deixa o treino bem mais lento)
import shutil
CORPUS = []
for p in [wiki] + pretrain.pastas_corpus(web):
    dst = Path("/content/corpus_local") / (p.parent.name + "_" + p.name)
    if not (dst / "meta.json").exists():
        shutil.copytree(p, dst, dirs_exist_ok=True)
    CORPUS.append(dst)
total = sum(json.loads((p / "meta.json").read_text())["tokens_treino"] for p in CORPUS)
print(f"corpus: {len(CORPUS)} parte(s), {total/1e6:,.0f} M tokens")

## Treino do dia

In [ ]:
# 5) Treina até completar a sessão (salva no Drive a cada 250 passos)
cfg = faisca_grande.CONFIG_FAISCA_200M
ck = Path(PASTA) / "pretreino.pt"
PASSOS_TOTAL = faisca_grande.TOKENS_ALVO // (faisca_grande.LOTE_SEQUENCIAS * cfg.block_size)
passo_inicial = torch.load(ck, map_location="cpu", weights_only=False)["passo"] if ck.exists() else 0
minutos = MINUTOS_SESSAO - (time.time() - INICIO) / 60 - 25
if TESTAR_AGORA or minutos < 10:
    minutos = 0.01
t0 = time.time()
res = pretrain.pretreinar(cfg, tok, CORPUS, ck, passos=PASSOS_TOTAL, lote=L["micro"], acumular=L["acumular"],
                          lr=4e-4, lr_min=4e-5, aquecimento=1000, tempo_max_min=minutos,
                          avaliar_cada=250, salvar_cada=250)
modelo = res["modelo"]

In [ ]:
# 6) Progresso e previsão
feitos = res["passo"] - passo_inicial
seg_passo = (time.time() - t0) / max(1, feitos)
prog_arq = Path(PASTA) / "progresso.json"
prog = json.loads(prog_arq.read_text()) if prog_arq.exists() else []
ult = res["hist"][-1] if res["hist"] else {}
if feitos > 0:
    prog.append({"data": time.strftime("%d/%m %H:%M"), "gpu": gpu, "passo": res["passo"], "passos_hoje": feitos,
                 "perda_val": ult.get("perda_val"), "ppl": ult.get("ppl_val"),
                 "tokens_M": round(res["passo"] * faisca_grande.LOTE_SEQUENCIAS * cfg.block_size / 1e6)})
    prog_arq.write_text(json.dumps(prog, indent=1))
for p in prog:
    print(p)
pct = res["passo"] / PASSOS_TOTAL
if feitos > 0:
    faltam = (PASSOS_TOTAL - res["passo"]) * seg_passo / 60 / (MINUTOS_SESSAO - 40)
    print(f"\nPROGRESSO: {pct:.1%} ({res['passo']:,}/{PASSOS_TOTAL:,} passos) | faltam ~{faltam:.0f} sessão(ões) como esta ({MINUTOS_SESSAO // 60} h) com esta GPU")
modelo.eval()
for inicio in ["O Brasil é", "A capital de Portugal", "Seis vezes oito é", "Para fazer um bolo"]:
    ids = [tok.doc] + tok.encode(inicio)
    print("→", inicio + tok.decode(modelo.gerar(ids, max_novos=40, temperatura=0.8, parar_em=(tok.fim, tok.doc),
                                                penalidade=1.2, sem_repetir=4)))
TERMINOU = res["passo"] >= PASSOS_TOTAL
FAZER_VERSAO = TERMINOU or TESTAR_AGORA
NOME_VERSAO = VERSAO if TERMINOU else f"{VERSAO}-p{round(pct * 100)}"
print("\n" + ("TREINO COMPLETO: seguindo para o ajuste de conversa e a avaliação." if TERMINOU else
              (f"PRÉVIA {NOME_VERSAO}: ajuste de conversa e avaliação agora; o pré-treino continua amanhã."
               if TESTAR_AGORA else "Sessão salva no Drive. Amanhã é só 'Executar tudo' de novo.")))

In [ ]:
# 6b) Régua fixa + cérebros guardados no Drive (não mexe no treino)
from heilo.training import regua
import shutil
REGUA = Path(PASTA) / "regua"
CEREBROS = Path("/content/drive/MyDrive/heilo_cerebros")
try:
    regua.congelar_r1([wiki, pretrain.pastas_corpus(web)[0]], REGUA)
    r_hoje = regua.medir(modelo, tok, REGUA, rotulo=f"Faísca 200M passo {res['passo']}")
    if feitos > 0 and prog:
        prog[-1]["regua"] = {k: r_hoje[k] for k in ("r1_ppl", "r2_acerto", "r2b_acerto") if k in r_hoje}
        prog_arq.write_text(json.dumps(prog, indent=1))
    for h in json.loads((REGUA / "historico.json").read_text())[-6:]:
        print(f"  {h['data']} | {h['rotulo']}: R1 {h['r1_ppl']} | R2 {h['r2_acerto']:.0%}"
              + (f" | R2b {h['r2b_acerto']:.0%}" if "r2b_acerto" in h else ""))
except Exception as e:
    print("a régua não mediu hoje (o treino não é afetado):", e)
if TERMINOU:
    # cópia congelada do fim da Fase 1: o pretreino.pt vai continuar mudando nas próximas fases
    CEREBROS.mkdir(parents=True, exist_ok=True)
    marco = CEREBROS / "fase1_faisca200m_final.pt"
    if not marco.exists():
        shutil.copy2(ck, marco)
        print("cópia congelada guardada em", marco)

## Versão (no fim do treino, ou prévia com TESTAR_AGORA = True)

In [ ]:
# 7) Ajuste de conversa + avaliação + GitHub
if FAZER_VERSAO:
    from heilo.training.pipeline import TrainingPipeline
    from heilo.training.ciclos import CiclosSeed
    from heilo.models.seed.gpt import MiniGPTChat
    pipe = TrainingPipeline()
    split = seed_v1.dividir_sft(pipe)
    print(split["manifest"])
    res_sft = pretrain.ajustar_conversa(modelo, split["treino"], split["val"], passos=3000, lote=L["micro"],
                                        lr=3e-5, paciencia=6, pasta_corpus=CORPUS, peso_texto=0.5,
                                        lote_texto=max(2, L["micro"] // 2))
    print("ajuste: guardado o passo", res_sft["passo_escolhido"], "| nota das respostas", res_sft["melhor_nota_geracao"])
    arq = pretrain.exportar(modelo, Path(PASTA) / f"heilo_seed_{NOME_VERSAO}.pt",
                            {"versao": NOME_VERSAO, "linha": "Faísca", "passos_pretreino": res["passo"],
                             "progresso": prog, "sft": res_sft})
    (CEREBROS / "versoes").mkdir(parents=True, exist_ok=True)
    shutil.copy2(arq, CEREBROS / "versoes" / Path(arq).name)   # toda versão fica guardada no Drive
    ciclos = CiclosSeed(pipe)
    dec = seed_v1.registrar(ciclos, arq, NOME_VERSAO, split["manifest"],
                            {"linha": "Faísca", "tamanho": "200M", "progresso": prog[-1] if prog else None,
                             "pretreino_pct": round(pct * 100, 1),
                             "sft": {k: res_sft[k] for k in ("perda_val_inicial", "melhor_perda_val",
                                                             "passo_escolhido", "melhor_nota_geracao")}},
                            origem=f"HEILO Faísca 200M (~210 M, tokenizer novo com algarismos separados), do zero "
                                   f"em sessões diárias ({pct:.0%} do pré-treino) + ajuste de conversa")
    print("PROMOVIDA" if dec["promovida"] else "NÃO promovida", "—", dec["motivo"])
    chat = MiniGPTChat(arquivo=ciclos.vdir / NOME_VERSAO / "heilo_seed.pt")
    for q in ["oi, tudo bem?", "quem é você?", "qual é a capital de portugal?", "quanto é 6 vezes 8?",
              "o que é uma função em python?", "o que faz o g04?", "me fala sobre o Brasil"]:
        print("Você:", q, "\nHEILO:", chat.responder([{"role": "user", "content": q}], temperatura=0.7,
                                                      penalidade=1.2, sem_repetir=4), "\n")
    add = aurora.preparar_para_git(Path("/content/heilo"))
    subprocess.run(["git", "-C", "/content/heilo", "config", "user.email", "heilo@colab"])
    subprocess.run(["git", "-C", "/content/heilo", "config", "user.name", "HEILO (Colab)"])
    subprocess.run(["git", "-C", "/content/heilo", "add", "-f"] + add + ["data/training/versions.json", "data/training/eval"])
    subprocess.run(["git", "-C", "/content/heilo", "commit", "-q", "-m", f"HEILO Faísca {NOME_VERSAO} 200M (Colab)"])
    r = subprocess.run("cd /content/heilo && git stash -u -q; git pull --rebase && git push", shell=True)
    print("ENVIADO PARA O GITHUB" if r.returncode == 0 else "ERRO no envio: mande um print desta célula")

## Fim: desliga a máquina (só depois que tudo foi salvo no Drive e no GitHub)

In [ ]:
# 8) Desliga o Colab para não gastar unidades à toa
if DESLIGAR_NO_FIM:
    # antes de desligar, espera o Drive terminar de enviar tudo (o pretreino.pt tem ~2,5 GB);
    # sem isso, a máquina pode sumir no meio do envio e o arquivo no Drive ficar velho ou incompleto
    print("Esperando o Drive terminar de salvar...")
    from google.colab import drive, runtime
    drive.flush_and_unmount()
    print("Tudo salvo no Drive. Desligando a máquina do Colab para parar de gastar unidades...")
    time.sleep(5)
    runtime.unassign()